# Energy Efficient GPU Computing - Tutorial

## 1. Hands-on: Mixed Precision

This notebook shows an example of how we can use **mixed precision** to improve the performance and energy efficiency of our GPU code. We'll again use **Kernel Tuner** to benchmark different code variants and [**Kernel Float**](https://kerneltuner.github.io/kernel_float) for simplified mixed-precision GPU programming.


### 1.1 Installing Dependencies
First we install `kernel_tuner`, `pynvml`, and `seaborn` using `pip`. The first lets us compile and benchmark GPU kernels from Python, while `pynvml` is needed to measure energy efficiency. `seaborn` is used for plotting. After installing, we import the packages we'll use.

In [ ]:
%%capture
!pip install kernel_tuner pynvml seaborn

import numpy as np
import matplotlib.pyplot as plt
import pandas
import seaborn
seaborn.set(context="notebook")

### 1.2 Installing Kernel Float

**Kernel Float** is header-only. It can be installed in several ways, but the
simplest is to download its single header file directly from GitHub, which we
do below.

In [ ]:
!wget -nc -O kernel_float.h https://raw.githubusercontent.com/KernelTuner/kernel_float/refs/heads/main/single_include/kernel_float.h

## 2. Mixed-Precision tuning

### 2.1 Problem Setup

In this hands-on session we'll tune a kernel that evaluates the Bessel
function of the first kind, order 0. This function appear throughout physics
in problems with cylindrical symmetry and is denoted $J_0$.

Below, we generate $n$ random inputs $x$ and compute the
reference values $J_0(x)$ with SciPy on the CPU.

In [ ]:
n = 40_000_000
input = np.random.rand(n) * 10.0

import scipy.special
expected_output = scipy.special.j0(input)

### 2.2 GPU kernel

Below is the GPU implementation of the Bessel function. It evaluates $J_0$ from its Taylor series. The kernel uses Kernel Float to make it possible to mix different precisions in the CUDA kernel.
Notice the four macros used in the kernel:

* `INPUT_TYPE`: data type used to store the input.
* `OUTPUT_TYPE`: data type used to store the output.
* `BLOCK_SIZE_X`: number of threads per block.
* `VECTOR_SIZE`: number of items per thread.

In [ ]:
%%writefile kernel.cu

#include "kernel_float.h"

extern "C"
__global__ void bessel(
  int n,
  const kernel_float::vec<INPUT_TYPE, VECTOR_SIZE>* input,
  kernel_float::vec<OUTPUT_TYPE, VECTOR_SIZE>* output
) {
  int global_index = blockIdx.x * BLOCK_SIZE_X + threadIdx.x;

  if (global_index * VECTOR_SIZE < n) {
    kernel_float::vec<double, VECTOR_SIZE> x = kernel_float::cast<double>(input[global_index]);
    kernel_float::vec<double, VECTOR_SIZE> z = (x * x) * kernel_float::constant(-0.25);
    kernel_float::vec<double, VECTOR_SIZE> t = 1;
    kernel_float::vec<double, VECTOR_SIZE> y = 1;

    #pragma unroll
    for (int k = 1; k <= 15; ++k) {
      t *= z * kernel_float::constant(1.0 / (k * k));
      y += t;
    }

    output[global_index] = kernel_float::cast<OUTPUT_TYPE>(y);
  }
}

### 2.3 Tuning the kernel

The cell below calls Kernel Tuner with the above kernel code. It builds up the pieces that `tune_kernel` needs:

* `tune_params`: the search space parameters.
* `args`: the kernel arguments.
* `answer`: the reference output to check corrrectness.
* `observers`: measure the numerical error and energy for each run.
* `strategy`: how to traverse the search space.
* `metrics`: derive metrics from the raw results.
* `compiler_options`: flags passed to the CUDA compiler.

In [ ]:
import kernel_tuner
from kernel_tuner.accuracy import TunablePrecision, AccuracyObserver
from kernel_tuner.observers.nvml import NVMLObserver

tune_params = dict()

# ============================================
# Tunable parameters.
# ✏️ Modify these to tune the storage types ✏️
# ============================================
# See assignment 1
tune_params["INPUT_TYPE"] = ["double"]
tune_params["OUTPUT_TYPE"] = ["double"]

# See assignment 2
# tune_params["COMPUTE_TYPE"] = ...

# See assignment 3
tune_params["BLOCK_SIZE_X"] = [256]
tune_params["VECTOR_SIZE"] = [1]

# ============================================
# Do not modify below this line!
# ============================================

# Kernel arguments wrapped in `TunablePrecision`s
args = [
    np.int32(n),
    TunablePrecision("INPUT_TYPE", input),
    TunablePrecision("OUTPUT_TYPE", np.zeros_like(expected_output)),
]

# The expected output
answer = [None, None, expected_output]

# Add observers to measure the accuracy and energy
observers = []
observers.append(AccuracyObserver("mean relative error"))
observers.append(NVMLObserver(["nvml_energy"]))

# Tune for 120 seconds using random sampling
strategy = "random_sample"
strategy_options = dict(time_limit=120, fraction=1)

# These compiler options are needed for compilation.
compiler_options=[
    "--std=c++17",
    "-I/content",
    "-I/usr/local/cuda/include",
]

# Derived metrics
metrics = dict()
metrics["logerror"] = lambda p: np.log10(p["error"])
metrics["energy"] = lambda p: p["nvml_energy"]
metrics["gflops"] = lambda p: (15 * 3 * n) / p["time"] * 1e-6

# Run kernel tuner!
results, env = kernel_tuner.tune_kernel(
    "bessel",
    "kernel.cu",
    n,
    args,
    tune_params,
    compiler_options=compiler_options,
    answer=answer,
    observers=observers,
    strategy=strategy,
    strategy_options=strategy_options,
    metrics=metrics,
    iterations=25,
    verbose=True,
)

## 3. Plotting the results

Kernel Tuner returns the results for every configuration in the search space. We can load these results into a _Pandas_ DataFrame and then plot them using _Seaborn_.

### 3.1 Analysis using Pandas

The cell below builds the DataFrame.

In [ ]:
show_columns = list(tune_params) + ["time", "error", "logerror", "energy", "gflops"]
time_baseline = 16.5  # Estimate of run-time on Tesla T4 in double precision
energy_baseline = 1.1 # Estimate of energy usage on Tesla T4 in double precision

df = pandas.DataFrame(results).filter(show_columns)
df["speedup"] = time_baseline / df["time"]
df["energy_reduction%"] = 100 * (1 - df["energy"] / energy_baseline)

def is_pareto(row, cols=["time", "logerror", "energy"]):
  any_better = (df < row).filter(cols).any(axis=1)
  no_worse = (df <= row).filter(cols).all(axis=1)
  return not (any_better & no_worse).any()
df["pareto"] = df.apply(is_pareto, axis=1)

df.sort_values("speedup", ascending=False)

### 3.2 Plotting using Seaborn

Next, we plot the results using Seaborn. The left plot shows speedup versus numerical error, while the right plot shows the enerage usage versus the numerical error.

In [ ]:
plt.subplots(1, 2, figsize=(15, 8))

# Time versus error
plt.subplot(121)
plt.title("time versus error")
plt.xlabel("Error")
plt.ylabel("Runtime (ms)")
plt.xscale("log")
plt.ylim(0, 20)
seaborn.scatterplot(df, x="error", y="time", hue="pareto")

# Speedup versus error
ax = plt.subplot(122)
plt.title("Speedup versus error")
plt.xlabel("Error")
plt.ylabel("Speedup over double precision")
plt.xscale("log")
seaborn.scatterplot(df, x="error", y="speedup", hue="pareto")

In [ ]:
plt.subplots(1, 2, figsize=(15, 8))

# Energy versus error
plt.subplot(121)
plt.title("Energy versus error")
plt.xlabel("Error")
plt.ylabel("Energy (Joule)")
plt.xscale("log")
seaborn.scatterplot(df, x="error", y="energy", hue="pareto")

# Energy reduction versus error
plt.subplot(122)
plt.title("Energy reduction versus error")
plt.xlabel("Error")
plt.ylabel("Energy reduction (%) over double precision")
plt.xscale("log")
plt.ylim(-5, 105)
seaborn.scatterplot(df, x="error", y="energy_reduction%", hue="pareto")

## 4. Assignments


### 4.1 Assignment 1
Make `INPUT_TYPE` and `OUTPUT_TYPE` **tunable**. Modify Cell 2.3 to change them from just `double` to also include `float` and `half`, then rerun all cells to see the effect.


### 4.2 Assignment 2
Changing only the storage types might not have the desired effect, since the kernel still computes in `double`. Add a tunable parameter `COMPUTE_TYPE` used for the internal computations. You need to do two things: (1) **modify the kernel code** in Cell 2.2 and (2) **modify `tune_params`** in Cell 2.3. Rerun all cells.


### 4.3 Assignment 3
Vectorization is currently disabled (`VECTOR_SIZE=1`). Make Kernel Tuner explore both `VECTOR_SIZE=1` and `VECTOR_SIZE=2`, then rerun the cells.


### 4.4 Assignment 4 (Bonus)
Currently, the number of iteration terms is fixed at 15 (See Cell 2.2), but lower precision needs fewer terms to converge (fewer precision bits). Add a tunable parameter `SERIES_TERMS` to tune the **number of terms**. Try smaller values like 10 or even 5. How low can you go?